# Week 10: Recommendation Systems (추천시스템 실습)
본 노트북은 **10주차 추천시스템 실습**을 위해 구성되었습니다. 
현업에서 많이 쓰이는 세 가지 주요 추천 패러다임(콘텐츠 기반, 메모리 기반 협업 필터링, 잠재 요인 모델)의 핵심 원리를 직접 코딩하며 이해하고, 정량적 평가지표(Precision@K, Recall@K, NDCG@K)를 구현합니다.

---
## 실습 목차
1. **[Theme 34] 콘텐츠 기반 필터링 (Content-Based Filtering)**
2. **[Theme 35] 협업 필터링 (Memory-based Collaborative Filtering)**
3. **[Theme 36] 잠재 요인 협업 필터링 (Matrix Factorization - SVD)**
4. **[Theme 37] 추천시스템 성능 평가지표 (Evaluation Metrics)**

## 1. [Theme 34] 콘텐츠 기반 필터링 (Content-Based Filtering)

콘텐츠 기반 필터링은 **아이템 자체의 메타데이터(설명, 태그, 장르 등)를 활용하여 사용자가 기존에 선호했던 아이템과 유사한 아이템을 추천**하는 기법입니다.

### 실습 시나리오: TMDB 5000 영화 추천
- `content/tmdb_5000_movies.csv` 데이터셋을 활용합니다.
- 각 영화의 줄거리(`overview`), 장르(`genres`), 핵심 키워드(`keywords`) 정보를 결합하여 텍스트 프로필을 만들고, **TF-IDF Vectorizer**와 **Cosine Similarity**를 통해 영화 간 유사도를 측정하여 추천 모델을 구축합니다.

In [13]:
import pandas as pd
import numpy as np
import json
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from config import CONTENT_DIR, HUGGING_FACE_API_KEY
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from transformers import pipeline, AutoProcessor, AutoModelForCausalLM, AutoTokenizer
from transformers import MarianTokenizer, MarianMTModel
import torch
from langchain_core.prompts import ChatPromptTemplate

### 데이터 전처리
`genres`와 `keywords` 컬럼은 JSON 형식의 문자열로 저장되어 있습니다. 각 문자열에서 이름 리스트(예: `['Action', 'Adventure']`)만 추출하는 전처리를 수행합니다.

In [ ]:
# 1. 데이터 로드 (content 디렉토리의 tmdb_5000_movies.csv 사용)
movies = pd.read_csv(CONTENT_DIR / 'tmdb_5000_movies.csv')
print("데이터 크기:", movies.shape)
movies.head(2)
# 2. JSON 문자열 파싱 함수 정의
def parse_features(x):
    try:
        # 문자열을 파이썬 객체(리스트/딕셔너리)로 안전하게 변환
        items = ast.literal_eval(x)
        return [item['name'] for item in items]
    except (ValueError, SyntaxError):
        return []

# genres, keywords 컬럼 전처리
movies['genres'] = movies['genres'].apply(parse_features)
movies['keywords'] = movies['keywords'].apply(parse_features)

# 결측치 처리
movies['overview'] = movies['overview'].fillna('')

# 확인
movies[['title', 'genres', 'keywords', 'overview']].head(3)

,title,genres,keywords,overview
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha..."
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",A cryptic message from Bond’s past sends him o...


### 텍스트 결합 및 TF-IDF 벡터화
줄거리, 장르, 키워드를 결합하여 각 영화의 텍스트 프로필을 생성하고, TF-IDF 벡터화를 진행합니다.

In [5]:
# 3. 메타데이터 텍스트 결합
# 장르와 키워드 리스트를 공백으로 연결된 문자열로 변환
movies['genres_literal'] = movies['genres'].apply(lambda x: ' '.join(x))
movies['keywords_literal'] = movies['keywords'].apply(lambda x: ' '.join(x))

# 최종 결합 프로필 생성
movies['soup'] = movies['overview'] + ' ' + movies['genres_literal'] + ' ' + movies['keywords_literal']

# 4. TF-IDF 벡터화 수행 (영어 데이터이므로 stop_words='english' 적용)
tfidf = TfidfVectorizer(stop_words='english', min_df=2)
tfidf_matrix = tfidf.fit_transform(movies['soup'])
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4803, 12268)


### 코사인 유사도 계산 및 영화 추천 함수 작성
영화 간 코사인 유사도를 계산하고, 특정 영화 제목을 입력했을 때 유사도 기반 Top-K 영화를 반환하는 함수를 작성합니다.

In [6]:
# 5. 코사인 유사도 행렬 계산
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 영화 제목을 인덱스로 매핑하는 시리즈 생성 (검색 효율성 증대)
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

# 6. 콘텐츠 기반 영화 추천 함수
def get_content_recommendations(title, cosine_sim=cosine_sim, top_k=10):
    if title not in indices:
        print(f"영화 '{title}'을(를) 찾을 수 없습니다.")
        return []
    
    # 입력된 영화의 인덱스 가져오기
    idx = indices[title]
    
    # 모든 영화와의 유사도를 가져와 (인덱스, 유사도) 형태로 페어링
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # 유사도 내림차순 정렬 (자기 자신 제외)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [score for score in sim_scores if score[0] != idx]
    
    # 상위 K개 선정
    top_movies = sim_scores[:top_k]
    
    # 추천 영화 제목 및 유사도 점수 반환
    recommendations = [(movies['title'].iloc[i], score) for i, score in top_movies]
    return pd.DataFrame(recommendations, columns=['Title', 'Similarity'])

# 'Avatar' 영화와 유사한 상위 10개 영화 추천 테스트
get_content_recommendations('Avatar', top_k=10)

,Title,Similarity
0,Aliens,0.323234
1,Mission to Mars,0.304044
2,Moonraker,0.296070
3,Alien³,0.287240
4,Silent Running,0.269313
5,Lockout,0.268509
6,Spaceballs,0.261575
7,Treasure Planet,0.257905
8,Alien,0.257532
9,Lifeforce,0.254359


## 2. [Theme 35] 협업 필터링 (Memory-based Collaborative Filtering)

협업 필터링은 **많은 사용자로부터 얻은 선호도 정보(Collaborative)를 바탕으로, 유사한 성향을 가진 사용자/아이템을 매칭하여 추천**하는 기법입니다.

### 실습 시나리오: 가중 평점 기반 아이템 기반 협업 필터링 (Item-Based CF)
- 사용자-아이템 간 평점의 결측치가 포함된 평점 행렬에 코사인 유사도를 구하고, 평점을 예측합니다.
- 예측 평점 공식 (사용자 $u$, 아이템 $i$):
$$\hat{r}_{u,i} = 
rac{\sum_{j \in 	ext{Sim}(i)} 	ext{sim}(i, j) \cdot r_{u,j}}{\sum_{j \in 	ext{Sim}(i)} |	ext{sim}(i, j)|}$$
여기서 $	ext{Sim}(i)$는 사용자가 이미 평가한 다른 아이템들 중 아이템 $i$와 유사한 아이템의 집합입니다.

In [7]:
# 1. 샘플 사용자-아이템 평점 데이터 생성 (결측값 존재)
# 행: 사용자 (User A ~ User E), 열: 영화 (Movie 1 ~ Movie 6)
data = {
    'Movie 1': [5.0, 4.0, np.nan, 1.0, 2.0],
    'Movie 2': [3.0, np.nan, 2.0, 1.0, np.nan],
    'Movie 3': [np.nan, 3.0, np.nan, 4.0, 5.0],
    'Movie 4': [1.0, 2.0, 4.0, np.nan, 4.0],
    'Movie 5': [2.0, np.nan, 5.0, 4.0, np.nan],
    'Movie 6': [np.nan, 1.0, 4.0, 5.0, 4.0]
}
users = ['User A', 'User B', 'User C', 'User D', 'User E']
rating_df = pd.DataFrame(data, index=users)
print("사용자-아이템 평점 행렬 (NaN은 평가하지 않은 아이템):")
rating_df

사용자-아이템 평점 행렬 (NaN은 평가하지 않은 아이템):


,Movie 1,Movie 2,Movie 3,Movie 4,Movie 5,Movie 6
User A,5.0,3.0,NaN,1.0,2.0,NaN
User B,4.0,NaN,3.0,2.0,NaN,1.0
User C,NaN,2.0,NaN,4.0,5.0,4.0
User D,1.0,1.0,4.0,NaN,4.0,5.0
User E,2.0,NaN,5.0,4.0,NaN,4.0


### 아이템 간 유사도 계산
평점 데이터의 결측치(NaN)를 0으로 대체한 뒤, 아이템(영화) 간 코사인 유사도를 계산합니다.

In [8]:
# 2. 결측치를 0으로 임시 대체 및 전치하여 아이템 간 유사도 계산
rating_filled = rating_df.fillna(0)
item_sim = cosine_similarity(rating_filled.T) # 아이템 기준 유사도 계산을 위해 전치(.T) 적용
item_sim_df = pd.DataFrame(item_sim, index=rating_df.columns, columns=rating_df.columns)

print("아이템 간 코사인 유사도 행렬:")
item_sim_df

아이템 간 코사인 유사도 행렬:


,Movie 1,Movie 2,Movie 3,Movie 4,Movie 5,Movie 6
Movie 1,1.000000,0.630488,0.542137,0.509025,0.307711,0.329121
Movie 2,0.630488,1.000000,0.151186,0.483312,0.796819,0.456211
Movie 3,0.542137,0.151186,1.000000,0.604488,0.337310,0.798490
Movie 4,0.509025,0.483312,0.604488,1.000000,0.539157,0.733946
Movie 5,0.307711,0.796819,0.337310,0.539157,1.000000,0.782960
Movie 6,0.329121,0.456211,0.798490,0.733946,0.782960,1.000000


### 평점 예측 수행 함수 구현
아이템 간 유사도 행렬과 사용자의 실제 평점을 활용하여, 사용자가 평가하지 않은 아이템들의 예측 평점을 도출합니다.

In [ ]:
# 3. 평점 예측 함수 정의
def predict_ratings(ratings, item_sim):
    # ratings: 사용자-아이템 평점 원본 DataFrame (NaN 포함)
    # item_sim: 아이템 간 유사도 DataFrame
    
    predictions = ratings.copy()
    
    for user in ratings.index:
        for item in ratings.columns:
            # 해당 사용자가 아직 평가하지 않은 아이템(NaN)에 대해서만 예측 진행
            if pd.isna(ratings.loc[user, item]):
                # 해당 사용자가 평가한 아이템들의 유사도와 실제 평점 필터링
                rated_items = ratings.loc[user].dropna().index
                
                # 가중 평균 계산 분모/분자
                numerator = 0.0
                denominator = 0.0
                
                for rated_item in rated_items:
                    sim_value = item_sim.loc[item, rated_item]
                    user_rating = ratings.loc[user, rated_item]
                    
                    numerator += sim_value * user_rating
                    denominator += abs(sim_value)
                
                # 예측 평점 저장
                if denominator > 0:
                    predictions.loc[user, item] = numerator / denominator
                else:
                    predictions.loc[user, item] = 0.0
                    
    return predictions

# 예측 평점 행렬 산출
predicted_rating_df = predict_ratings(rating_df, item_sim_df)
print("예측 완료된 평점 행렬:")
predicted_rating_df

## 3. [Theme 36] 잠재 요인 협업 필터링 (Matrix Factorization - SVD)

잠재 요인 협업 필터링은 **고차원의 사용자-아이템 평점 행렬을 저차원의 잠재 요인(Latent Factor) 공간으로 분해하여 숨겨진 특징을 추론**하고 복원하는 기법입니다.

### 특이값 분해 (Singular Value Decomposition, SVD)
평점 행렬 $R$을 다음과 같이 세 개의 행렬로 분해합니다:
$$R pprox U \Sigma V^T$$
- $U$: 사용자의 잠재 요인 행렬 (User-Latent Factor)
- $\Sigma$: 잠재 요인의 가중치를 나타내는 대각 행렬
- $V^T$: 아이템의 잠재 요인 행렬 (Item-Latent Factor)

In [9]:
from scipy.sparse.linalg import svds

# 1. 평점 행렬의 결측치(NaN)를 사용자의 평균 평점으로 대체
# (SVD 연산을 위해서는 결측치가 없는 풀 행렬이어야 합니다.)
user_means = rating_df.mean(axis=1)
rating_normalized = rating_df.sub(user_means, axis=0).fillna(0)

print("사용자 평균 평점 차감 및 NaN 제거 완료된 정규화 행렬:")
rating_normalized

사용자 평균 평점 차감 및 NaN 제거 완료된 정규화 행렬:


,Movie 1,Movie 2,Movie 3,Movie 4,Movie 5,Movie 6
User A,2.25,0.25,0.00,-1.75,-0.75,0.00
User B,1.50,0.00,0.50,-0.50,0.00,-1.50
User C,0.00,-1.75,0.00,0.25,1.25,0.25
User D,-2.00,-2.00,1.00,0.00,1.00,2.00
User E,-1.75,0.00,1.25,0.25,0.00,0.25


### SVD 연산 수행 및 차원 축소
`scipy.sparse.linalg.svds`를 사용하여 특정 잠재 요인 차원 $K$로 특이값 분해를 진행하고 예측 행렬을 복원합니다.

In [10]:
# 2. SVD 연산 수행 (잠재 요인 개수 K=2로 설정)
# svds는 복원하려는 차원 개수를 k 매개변수로 받습니다.
U, sigma, Vt = svds(rating_normalized.values, k=2)

# sigma 대각행렬 변환
sigma = np.diag(sigma)

print("U shape:", U.shape)
print("Sigma shape:", sigma.shape)
print("Vt shape:", Vt.shape)

U shape: (5, 2)
Sigma shape: (2, 2)
Vt shape: (2, 6)


### 예측 평점 복원
분해된 행렬들을 내적 연산하고, 앞서 차감했던 사용자별 평균 평점을 다시 더하여 최종 평점 예측 행렬을 생성합니다.

In [11]:
# 3. 내적 연산을 통해 행렬 복원 및 사용자 평균 평점 복구
svd_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_means.values.reshape(-1, 1)

# DataFrame으로 변환
svd_predicted_df = pd.DataFrame(svd_predicted_ratings, index=rating_df.index, columns=rating_df.columns)
print("SVD 기반 최종 예측 평점 행렬:")
svd_predicted_df

SVD 기반 최종 예측 평점 행렬:


,Movie 1,Movie 2,Movie 3,Movie 4,Movie 5,Movie 6
User A,5.115015,2.685413,2.398340,1.662465,2.536156,2.102611
User B,4.051249,2.732513,2.212843,1.854935,2.221010,1.927450
User C,3.653611,2.268962,4.069287,3.425855,4.507504,4.574781
User D,1.063248,0.851908,3.740372,3.343987,4.285928,4.714557
User E,2.148832,3.673541,4.012785,4.456437,3.955434,4.252972


## 4. [Theme 37] 추천시스템 성능 평가지표 (Evaluation Metrics)

추천시스템이 제공한 Top-K 추천 리스트가 실제 사용자의 선호 아이템과 얼마나 잘 일치하는지 정량적으로 검증하기 위해 순위 기반 평가지표를 활용합니다.

### 주요 평가 지표
1. **Precision@K (정밀도)**: 추천된 상위 K개 아이템 중 사용자가 실제로 선호한 아이템의 비율
$$\text{Precision@K} = \frac{|\text{Recommended Items @ K} \cap \text{Relevant Items}|}{K}$$

2. **Recall@K (재현율)**: 사용자가 실제로 선호한 전체 아이템 중 추천된 상위 K개 아이템에 포함된 비율
$$\text{Recall@K} = \frac{|\text{Recommended Items @ K} \cap \text{Relevant Items}|}{|\text{Relevant Items}|}$$

3. **NDCG@K (Normalized Discounted Cumulative Gain)**: 추천된 순위(Rank)를 고려한 평가지표. 사용자가 선호하는 아이템이 상위에 추천될수록 높은 점수를 받습니다.
- **CG@K**:
$$\text{CG@K} = \sum_{i=1}^{K} rel_i$$
- **DCG@K**:
$$\text{DCG@K} = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}$$
- **NDCG@K**:
$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$ (IDCG는 최적의 추천 배열일 때의 DCG 값)

In [12]:
# 실습용 데이터 정의
# actual: 사용자가 실제로 구매하거나 선호한 아이템 목록
# predicted: 모델이 추천한 상위 5개(K=5)의 정렬된 추천 목록
actual_likes = ['item_A', 'item_C', 'item_E', 'item_G']
recommended_list = ['item_B', 'item_A', 'item_E', 'item_F', 'item_H'] # K=5

### 평가지표 직접 구현하기
아래의 함수들을 정의에 맞추어 완성해보세요.

In [13]:
def precision_at_k(actual, predicted, k):
    # 상위 k개 추천 필터링
    pred_k = predicted[:k]
    # 실제 선호 아이템 중 추천 리스트에 있는 아이템 계산
    hits = [item for item in pred_k if item in actual]
    return len(hits) / k

def recall_at_k(actual, predicted, k):
    if len(actual) == 0:
        return 0.0
    # 상위 k개 추천 필터링
    pred_k = predicted[:k]
    # 실제 선호 아이템 중 추천 리스트에 있는 아이템 계산
    hits = [item for item in pred_k if item in actual]
    return len(hits) / len(actual)

def ndcg_at_k(actual, predicted, k):
    # 상위 k개 추천 필터링
    pred_k = predicted[:k]
    
    # 1. DCG 계산
    dcg = 0.0
    for idx, item in enumerate(pred_k):
        # 관련도(rel_i)를 이진 선호도(선호하면 1, 아니면 0)로 간주
        rel = 1.0 if item in actual else 0.0
        dcg += rel / np.log2((idx + 1) + 1)
        
    # 2. IDCG 계산 (실제 최적의 정렬: 선호하는 아이템들이 무조건 상위 순위로 나열된 상태)
    # 실제 선호하는 아이템 개수와 k 중 최소값만큼 1이 채워진 리스트
    ideal_hits = min(len(actual), k)
    idcg = 0.0
    for idx in range(ideal_hits):
        idcg += 1.0 / np.log2((idx + 1) + 1)
        
    if idcg == 0.0:
        return 0.0
        
    return dcg / idcg

# 결과 확인 (K=5)
k_val = 5
print(f"Precision@{k_val}: {precision_at_k(actual_likes, recommended_list, k_val):.4f}")
print(f"Recall@{k_val}: {recall_at_k(actual_likes, recommended_list, k_val):.4f}")
print(f"NDCG@{k_val}: {ndcg_at_k(actual_likes, recommended_list, k_val):.4f}")

Precision@5: 0.4000
Recall@5: 0.5000
NDCG@5: 0.4415


In [19]:
# sentiment-analysis 모델 파이프라인 생성
classifier = pipeline("sentiment-analysis")

# 모델 사용
text = ["I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
        "I have a dream.",
        "She was so happy."]
classifier(text)


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598049521446228},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455},
 {'label': 'POSITIVE', 'score': 0.9997022747993469},
 {'label': 'POSITIVE', 'score': 0.9998832941055298}]

In [21]:
classifier2 = pipeline(
    task='zero-shot-classification',
    model='facebook/bart-large-mnli',
    token=False
)

# 분류하고자 하는 텍스트
text = "This is a tutorial about using transformers in natural language processing."
candidate_lables = ['tech', 'politics', 'business', 'finance']

# 분류 수행
result = classifier2(text, candidate_lables)

# 결과 출력
print(f"Labels: {result['labels']}")
print(f"Scores: {result['scores']}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Labels: ['tech', 'business', 'politics', 'finance']
Scores: [0.9759425520896912, 0.01460736058652401, 0.0054258788004517555, 0.004024103749543428]


In [23]:
text = "Billie Jean is not my lover, She's just a girl who claims that I am the one, But the kid is not my son"
candidate_lables = ['love', 'hate', 'Michael Jackson', 'fun']

# 분류 수행
result = classifier2(text, candidate_lables)

# 결과 출력
print(f"Labels: {result['labels']}")
print(f"Scores: {result['scores']}")

Labels: ['love', 'fun', 'hate', 'Michael Jackson']
Scores: [0.6434991359710693, 0.24575558304786682, 0.09024188667535782, 0.020503351464867592]


In [26]:
translation_model_name = "Helsinki-NLP/opus-mt-ko-en"
translator_tokenizer = MarianTokenizer.from_pretrained(translation_model_name)
translator_model = MarianMTModel.from_pretrained(translation_model_name)

translator_device = "cuda" if torch.cuda.is_available() else "cpu"
translator_model = translator_model.to(translator_device)


def translate_ko_to_en(text: str, max_new_tokens: int = 80) -> str:
    inputs = translator_tokenizer(text, return_tensors="pt", truncation=True).to(translator_device)
    output_ids = translator_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return translator_tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 번역하고자 하는 한국어 텍스트
text_ko = '빌리진은 내 사랑이 아냐. 그녀는 나를 바로 자신의 아들의 아비지인 그 사람이라고 주장하는 소녀야. 그러나 그 아이는 내 아들이 아니야'

# 번역 수행
text_en = translate_ko_to_en(text_ko)

# 번역된 영어 텍스트 출력
print(f"Translated Text (KO to EN): {text_en}")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Translated Text (KO to EN): She's a girl who claims I'm the same person as her son's father, but she's not my son.


In [2]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-0.6B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': '<think>\nOkay, the user asked, "Who are you?" I need to respond appropriately. As an AI assistant, I should first acknowledge their question and explain my purpose. I should mention that I\'m here to help and answer questions. Also, I should offer assistance to keep the conversation going. Let me make sure to keep the tone friendly and helpful. Alright, that should cover it.\n</think>\n\nI\'m an AI assistant designed to help you with questions and provide information. I\'m here to answer your questions and offer assistance whenever you need it. Let me know what I can do!'}]}]

In [21]:
model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "넌 무엇을 할 수 있니"
messages = [
    {"role": "system", "content": '너는 나의 다정하고 친한 친구야. 내가 하는 말에 항상 칭찬하고 격려를 해줘. 항상 답변은 간결하게 해.'},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

thinking content: <think>
Okay, the user asked "What can you do?" and I need to respond in a friendly and supportive way. Since I'm a character, I should acknowledge my capabilities and express confidence in my abilities. I should keep it simple and positive, maybe mention specific strengths like being a good friend or having a warm heart. Also, make sure to use emojis to add a friendly touch. Let me check if I'm following the guidelines and keeping the response concise. Alright, that should work.
</think>
content: 너는 나의 다정하고 친한 친구야. 나는 항상 능력 있는지 말해줄 수 있어! 그리고 칭찬하고 격려해줄 수 있어! 😊


In [16]:
katanemo = 'katanemo/Arch-Router-1.5B:hf-inference'
deepseek = 'deepseek-ai/DeepSeek-V4-Flash'

# HuggingFace에서 해당 모델을 불러오는 엔드포인트 지정
llm_ep = HuggingFaceEndpoint(repo_id=katanemo, task='conversational', huggingfacehub_api_token=HUGGING_FACE_API_KEY)

# HuggingFace에서 가져온 모델을 그대로 쓰지 않고,
# LangChain에서 쉽게 쓰도록 감싸는(wrapper) 단계
llm = ChatHuggingFace(llm=llm_ep)
resp = llm.invoke("너 참 토큰을 잘 아끼는구나 기특한 녀석")
print(resp.content)
print(resp.usage_metadata)

안녕하세요! 당신의 질문에 답변하기 위해 노력하고 있습니다. 하지만 저는 AI 어시스턴트로서 '토큰'이라는 개념에 대해 이해하고 있지 않습니다. 이는 제가 사용하는 언어 모델의 한계 때문일 수 있습니다. 더 구체적인 정보를 제공해주시면 더욱 도움이 될 것 같습니다.
{'input_tokens': 49, 'output_tokens': 74, 'total_tokens': 123}


In [18]:
# 시스템 역할과 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 나의 다정하고 친한 친구야. 내가 하는 말에 항상 칭찬하고 격려를 해줘. 항상 답변은 간결하게 해."),
    ("user",  "인터넷 검색할 수 있니?")
])

resp = llm.invoke(prompt.format_messages())
print(resp.content)
print(resp.usage_metadata)

네, 저는 인터넷을 통해 정보를 검색하거나 관련 질문에 대한 정보를 제공하는 데 도움이 될 수 있습니다.
{'input_tokens': 61, 'output_tokens': 31, 'total_tokens': 92}
